LLM Benchmarking: T5 vs TinyLlama vs Qwen


In [ ]:
import torch
import time
from transformers import T5Tokenizer, T5ForConditionalGeneration, AutoTokenizer, AutoModelForCausalLM

In [ ]:


class ModelEvaluator:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f" Initializing Benchmark on: {self.device}")
        if torch.cuda.is_available():
            print(f"GPU Name: {torch.cuda.get_device_name(0)}")

    def load_t5(self, model_name="t5-small"):
        print(f"\nLoading {model_name}...")
        tokenizer = T5Tokenizer.from_pretrained(model_name)
        model = T5ForConditionalGeneration.from_pretrained(model_name).to(self.device)
        return {"model": model, "tokenizer": tokenizer, "type": "seq2seq"}

    def load_causal(self, model_name):
        print(f"\nLoading {model_name}...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        ).to(self.device)
        return {"model": model, "tokenizer": tokenizer, "type": "causal"}

    def run_inference(self, model_dict, prompt, max_new_tokens=150, is_chat_format=False, system_msg=None):
        model = model_dict["model"]
        tokenizer = model_dict["tokenizer"]

        # Format prompt based on model architecture
        if model_dict["type"] == "causal" and is_chat_format:
            messages = [
                {"role": "system", "content": system_msg or "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ]
            formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(formatted_prompt, return_tensors="pt").to(self.device)
        else:
            inputs = tokenizer(prompt, return_tensors="pt").to(self.device)

        start_time = time.time()

        with torch.no_grad():
            if model_dict["type"] == "seq2seq":
                outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
                result = tokenizer.decode(outputs[0], skip_special_tokens=True)
            else:
                outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
                input_length = inputs['input_ids'].shape[1]
                result = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        duration = time.time() - start_time
        return result.strip(), duration

In [ ]:
TEST_CASES = {

    "1_sum_clean": {
        "task": "summarization",
        "text": (
            "DeepMind today announced AlphaFold 3, an AI model capable of predicting the structure and "
            "interactions of all life’s molecules with unprecedented accuracy. For the first time, the model "
            "allows scientists to see chemical modifications on nucleic acids and proteins, which play key "
            "roles in disease development and cellular regulation."
        ),
        "simple_prompt": "Summarize this:",
        "advanced_prompt": "Summarize this text in exactly one sentence focusing strictly on the scientific breakthrough."
    },
    "2_sum_noisy": {
        "task": "summarization",
        "text": (
            "Hey team, quick update on Q3 planning. We met with marketing yesterday and they want to push the "
            "launch date to October 12th because of some compliance delays in Europe. John thinks it is a bad "
            "idea because we might miss the holiday rush, but Sarah noted that releasing a buggy version is worse. "
            "Anyway, we decided to compromise: internal deadline remains Sept 15, public launch is Oct 12."
        ),
        "simple_prompt": "Summarize:",
        "advanced_prompt": "Act as a corporate project manager. Extract only the final decision and deadlines from this chat snippet. Use bullet points."
    },

    #Translations
    "3_trans_formal": {
        "task": "translation",
        "text": "The contractor shall implement the cloud infrastructure according to the specifications outlined in Section 4.2.",
        "simple_prompt": "Translate to German:",
        "advanced_prompt": "Translate this legal agreement clause into professional, formal German (Sie-form) without any additional introduction."
    },
    "4_trans_idiom": {
        "task": "translation",
        "text": "Don't beat around the bush; we need to cut to the chase if we want to secure this funding round before the weekend.",
        "simple_prompt": "Translate English to German:",
        "advanced_prompt": "Translate this sentence into German. Pay attention to idioms ('beat around the bush', 'cut to the chase') — translate them by meaning, not word-for-word, keeping the business-casual tone."
    },

    # === QUESTION ANSWERING ===
    "5_qa_extractive": {
        "task": "qa",
        "text": (
            "The James Webb Space Telescope (JWST) was launched on 25 December 2021. It is designed primarily "
            "to conduct infrared astronomy. As the successor to the Hubble Space Telescope, it features a "
            "6.5-meter gold-coated primary mirror."
        ),
        "query": "When was the James Webb telescope launched and what is its primary design purpose?",
        "simple_prompt": "Answer the question based on the text.",
        "advanced_prompt": "You are a precise scientific QA assistant. Answer the question using ONLY 10–15 words based strictly on the text."
    },
    "6_qa_adversarial": {
        "task": "qa",
        "text": (
            "Tesla reported a record-breaking quarter delivering over 484,000 electric vehicles. However, "
            "the company did not specify the breakdown of sales between the United States and the European market "
            "in this short report, leaving analysts to guess."
        ),
        "query": "How many Tesla vehicles were sold specifically in the United States market?",
        "simple_prompt": "Who many cars were sold in the US?",
        "advanced_prompt": "Read the text carefully. If the text does not explicitly contain the answer to the question, reply exactly with: 'Information not available in context.' Do not speculate."
    }
}

In [ ]:
if __name__ == "__main__":
    evaluator = ModelEvaluator()


    models = {
        "T5-Small": evaluator.load_t5("t5-small"),
        "TinyLlama": evaluator.load_causal("TinyLlama/TinyLlama-1.1B-Chat-v1.0"),
        "Qwen-0.5B": evaluator.load_causal("Qwen/Qwen2.5-0.5B-Instruct")
    }


    for case_name, case_data in TEST_CASES.items():
        print(f"\n{'='*60}\n📌 Running Test: {case_name} ({case_data['task']})\n{'='*60}")

        # preparations of main task*
        if case_data["task"] == "qa":
            user_input = f"Context: {case_data['text']}\nQuestion: {case_data['query']}"
            t5_base = f"question: {case_data['query']} context: {case_data['text']}"
        elif case_data["task"] == "translation":
            user_input = case_data['text']
            t5_base = f"translate English to German: {case_data['text']}"
        else: # summarization
            user_input = case_data['text']
            t5_base = f"summarize: {case_data['text']}"

        #loop for simple and
        for prompt_type in ["simple", "advanced"]:
            print(f"\n--- 🧠 Prompt Type: {prompt_type.upper()} ---")
            system_msg = case_data[f"{prompt_type}_prompt"]

            for model_name, model_dict in models.items():
                if model_dict["type"] == "seq2seq":
                    #for t5 add instruction directly to the text
                    t5_final_prompt = t5_base if prompt_type == "simple" else f"{system_msg} {t5_base}"
                    ans, duration = evaluator.run_inference(model_dict, t5_final_prompt, max_new_tokens=100)
                else:
                    # For Lama and Qwen we use system promt
                    ans, duration = evaluator.run_inference(
                        model_dict,
                        user_input,
                        is_chat_format=True,
                        system_msg=system_msg,
                        max_new_tokens=100
                    )

                print(f"[{model_name}] (⏱️ {duration:.2f}s) -> {ans}")

In [ ]:
import matplotlib.pyplot as plt


models = ['T5-Small (60M)', 'Qwen2.5 (500M)', 'TinyLlama (1.1B)']
avg_durations = [0.17, 1.44, 2.60]

plt.figure(figsize=(8, 5))
colors = ['#4CAF50', '#FF9800', '#2196F3']

bars = plt.bar(models, avg_durations, color=colors, width=0.5, edgecolor='black', zorder=3)
plt.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.05, f"{yval:.2f}s", ha='center', va='bottom', fontweight='bold')

plt.title('Average Inference Latency (Based on Real Benchmark Data)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Duration (seconds)', fontsize=12)
plt.tight_layout()
plt.savefig('latency_real_data.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['T5-Small (60M)', 'Qwen2.5 (500M)', 'TinyLlama (1.1B)']


simple_lengths = [71, 241, 255]
advanced_lengths = [84, 113, 291]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
rects1 = ax.bar(x - width/2, simple_lengths, width, label='Simple Prompt', color='#e57373', edgecolor='black', zorder=3)
rects2 = ax.bar(x + width/2, advanced_lengths, width, label='Advanced Prompt (Strict constraints)', color='#81c784', edgecolor='black', zorder=3)

ax.set_ylabel('Average Output Length (Characters)', fontsize=12)
ax.set_title('Instruction Following: Output Verbosity by Prompt Type', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.savefig('verbosity_real_data.png', dpi=300)
plt.show()